# Workflow Testing Notebook

This notebook helps you test and improve your multi-agent workflow:
- Send a message through the full orchestrator
- Receive the final reply
- Inspect all workflow steps from the tracker
- Run each agent separately to compare quality and latency

In [1]:
import asyncio
import os
import sys
from datetime import datetime
from pathlib import Path
from time import perf_counter

import pandas as pd
from IPython.display import Markdown, display

try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None

def _resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for p in candidates:
        if (p / "pyproject.toml").exists() and (p / "src").exists():
            return p
    return Path.cwd()

PROJECT_ROOT = _resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Keep all relative paths (for secrets/docs) stable regardless of where notebook starts.
os.chdir(PROJECT_ROOT)

if load_dotenv is not None:
    load_dotenv(PROJECT_ROOT / ".env")

print(f"Project root: {PROJECT_ROOT}")
print(f"Current working directory: {Path.cwd()}")
print("Tip: make sure required API keys are set in environment/.env before running tests.")

Project root: /Users/nelsoncamoes/dev/personal/worldcup_2026
Current working directory: /Users/nelsoncamoes/dev/personal/worldcup_2026
Tip: make sure required API keys are set in environment/.env before running tests.


In [2]:
import importlib
import src.agents.orchestrator as _orch_mod
import src.agents.workflow_logger as _log_mod
import src.agents.planner_agent as _planner_mod
import src.agents.bigquery_agent as _bq_mod

importlib.reload(_log_mod)
importlib.reload(_planner_mod)
importlib.reload(_bq_mod)
importlib.reload(_orch_mod)

from src.agents.orchestrator import run_orchestrator
from src.agents.workflow_logger import get_tracker, reset_tracker

from src.agents.news_agent import run_structured as run_news
from src.agents.sentiment_agent import run_structured as run_sentiment
from src.agents.prediction_agent import run_structured as run_prediction
from src.agents.bigquery_agent import run_structured as run_bigquery

from langchain_openai import ChatOpenAI

print("Orchestrator + planner + bigquery reloaded.")

/Users/nelsoncamoes/dev/personal/worldcup_2026/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Orchestrator + planner + bigquery reloaded.


In [3]:
def _run_async(coro):
    """Runs async code safely from notebook context.

    In Jupyter, an event loop is often already running. In that case, run the
    coroutine in a separate thread with its own loop.
    """
    import concurrent.futures

    try:
        return asyncio.run(coro)
    except RuntimeError as exc:
        if "asyncio.run() cannot be called from a running event loop" not in str(exc):
            raise

        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
            future = executor.submit(lambda: asyncio.run(coro))
            return future.result()


def run_workflow_test(
    user_message: str,
    user_id: str = "notebook_user",
    conversation_history: list[dict[str, str]] | None = None,
) -> dict:
    """Run the full orchestrator and return reply + workflow trace."""
    reset_tracker()
    t0 = perf_counter()
    reply = _run_async(
        run_orchestrator(
            user_message=user_message,
            user_id=user_id,
            conversation_history=conversation_history or [],
        )
    )
    total_ms = (perf_counter() - t0) * 1000

    steps = list(get_tracker().steps)
    return {
        "message": user_message,
        "reply": reply,
        "total_ms": round(total_ms, 2),
        "steps": steps,
    }


# ── Display helpers ───────────────────────────────────────────────────────

_STEP_ICONS = {
    "classify":        "🔎",
    "router":          "🗺️",
    "agent_execution": "⚙️",
    "aggregate":       "🔗",
    "confidence":      "📊",
    "compose":         "✍️",
}


def _hr():
    display(Markdown("---"))


def _section(title: str):
    display(Markdown(f"#### {title}"))


def _kv(label: str, value, indent: int = 0):
    pad = "&nbsp;" * (indent * 4)
    display(Markdown(f"{pad}**{label}:** {str(value)}"))


def _show_code_block(code: str, language: str = "sql"):
    display(Markdown(f"```{language}\n{code}\n```"))


def _show_nested(label: str, value, indent: int = 0):
    pad = "&nbsp;" * (indent * 4)

    if isinstance(value, dict):
        if not value:
            _kv(label, "{}", indent)
            return
        display(Markdown(f"{pad}**{label}:**"))
        for sub_key, sub_value in value.items():
            _show_nested(str(sub_key), sub_value, indent + 1)
        return

    if isinstance(value, list):
        if not value:
            _kv(label, "[]", indent)
            return
        if all(not isinstance(item, (dict, list)) for item in value):
            _kv(label, ", ".join(str(item) for item in value), indent)
            return
        display(Markdown(f"{pad}**{label}:**"))
        for idx, item in enumerate(value, 1):
            if isinstance(item, dict):
                display(Markdown(f"{'&nbsp;' * ((indent + 1) * 4)}**item {idx}:**"))
                for sub_key, sub_value in item.items():
                    _show_nested(str(sub_key), sub_value, indent + 2)
            else:
                _kv(f"item {idx}", item, indent + 1)
        return

    if label == "sql":
        display(Markdown(f"{pad}**sql:**"))
        _show_code_block(str(value), language="sql")
        return

    _kv(label, value, indent)


def _show_agent_outputs(agent_outputs: dict):
    for agent_name, payload in agent_outputs.items():
        if not isinstance(payload, dict):
            continue
        display(Markdown(f"**→ `{agent_name}`**"))
        _kv("data_source", payload.get("data_source", "unknown"), indent=1)
        _kv("confidence_score", payload.get("confidence_score"), indent=1)
        _kv("confidence_reason", payload.get("confidence_reason", ""), indent=1)

        metadata = payload.get("metadata") or {}
        if metadata:
            _show_nested("metadata", metadata, indent=1)

        answer = str(payload.get("answer", "")).strip()
        display(Markdown(f"&nbsp;&nbsp;&nbsp;&nbsp;**answer:**"))
        display(Markdown(answer or "_empty_"))


def _show_step(step: dict, delta_ms: float | None):
    node = step.get("node", "unknown")
    status = step.get("status", "")
    icon = _STEP_ICONS.get(node, "•")
    timing = f"  `+{delta_ms:.0f} ms`" if delta_ms is not None else ""
    display(Markdown(f"### {icon} `{node}` — {status}{timing}"))

    inp = step.get("input") or {}
    out = step.get("output") or {}
    meta = step.get("metadata") or {}

    if inp:
        _section("Inputs")
        for k, v in inp.items():
            if k == "user_message":
                display(Markdown("**user_message:**"))
                display(Markdown(f"> {v}"))
            else:
                _show_nested(k, v)

    if out:
        _section("Outputs")
        for k, v in out.items():
            if k == "agent_outputs":
                display(Markdown("**Per-agent results:**"))
                _show_agent_outputs(v)
            elif k in ("final_reply", "merged_answer", "answer"):
                display(Markdown(f"**{k}:**"))
                display(Markdown(str(v).strip()))
            else:
                _show_nested(k, v)

    if meta:
        _section("Metadata")
        for k, v in meta.items():
            _show_nested(k, v)

    _hr()


def show_workflow_result(result: dict) -> None:
    display(Markdown("# ✅ Final Reply"))
    display(Markdown(result["reply"]))
    display(Markdown(f"**Total latency:** `{result['total_ms']} ms`"))
    _hr()

    display(Markdown("# 🔬 Workflow Steps — Full Detail"))
    steps = result["steps"]
    prev_ts = None

    for step in steps:
        ts_raw = step.get("timestamp", "")
        ts = None
        if ts_raw:
            try:
                ts = datetime.fromisoformat(ts_raw)
            except Exception:
                pass

        delta_ms = None
        if ts is not None and prev_ts is not None:
            delta_ms = (ts - prev_ts).total_seconds() * 1000

        _show_step(step, delta_ms)

        if ts is not None:
            prev_ts = ts

In [4]:
# Full workflow test example
test_message = "What was Portugal vs Morocco last result and stats?"
result = run_workflow_test(test_message, user_id="notebook_user")
show_workflow_result(result)

# ✅ Final Reply

## Portugal vs Morocco Last Result

- **Date**: December 10, 2022
- **Competition**: World Cup, Quarter-finals
- **Venue**: Al Thumama Stadium, Doha
- **Final Score**: Morocco 1 - 0 Portugal
- **Referee**: F. Tello
- **Status**: Full Time

## Match Stats

### Morocco
- **Shots on Goal**: 3
- **Ball Possession**: 27%
- **Shots Off Goal**: 6
- **Total Shots**: 9
- **Corner Kicks**: 3
- **Fouls**: 15

### Portugal
- **Shots on Goal**: 3
- **Ball Possession**: 73%
- **Shots Off Goal**: 6
- **Total Shots**: 12
- **Corner Kicks**: 9
- **Fouls**: 9

Confidence: HIGH (80%)

**Total latency:** `19661.33 ms`

---

# 🔬 Workflow Steps — Full Detail

...

**intent:** data

#### Outputs

**final_reply:**

## Portugal vs Morocco Last Result

- **Date**: December 10, 2022
- **Competition**: World Cup, Quarter-finals
- **Venue**: Al Thumama Stadium, Doha
- **Final Score**: Morocco 1 - 0 Portugal
- **Referee**: F. Tello
- **Status**: Full Time

## Match Stats

### Morocco
- **Shots on Goal**: 3
- **Ball Possession**: 27%
- **Shots Off Goal**: 6
- **Total Shots**: 9
- **Corner Kicks**: 3
- **Fouls**: 15

### Portugal
- **Shots on Goal**: 3
- **Ball Possession**: 73%
- **Shots Off Goal**: 6
- **Total Shots**: 12
- **Corner Kicks**: 9
- **Fouls**: 9

Confidence: HIGH (80%)

---

In [ ]:
# Optional conversation history simulation
conversation = [
    {"role": "user", "content": "Show me Portugal next fixture"},
    {"role": "assistant", "content": "Portugal plays Team X on ..."},
]
follow_up = "And what are the win probabilities for that one?"
follow_up_result = run_workflow_test(follow_up, conversation_history=conversation)
show_workflow_result(follow_up_result)

In [ ]:
# match_facts_agent is no longer routed by the orchestrator.
# bigquery_agent is now the single source of truth for all structured data.
AGENT_RUNNERS = {
    "news": run_news,
    "sentiment": run_sentiment,
    "prediction": run_prediction,
    "bigquery": run_bigquery,
}

def run_chat_agent(query: str) -> dict:
    """Simple isolated chat baseline using the same LLM family."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    prompt = (
        "You are a helpful football assistant. Answer concisely and clearly.\n"
        f"User: {query}"
    )
    answer = llm.invoke(prompt).content.strip()
    return {
        "answer": answer,
        "confidence_score": 0.7,
        "confidence_reason": "Direct chat baseline response.",
        "metadata": {"path": "isolated_chat"},
    }

AGENT_RUNNERS["chat"] = run_chat_agent

def run_single_agent_test(agent_name: str, query: str) -> dict:
    if agent_name not in AGENT_RUNNERS:
        raise ValueError(f"Unknown agent: {agent_name}. Available: {list(AGENT_RUNNERS)}")

    t0 = perf_counter()
    error = None
    payload = {}
    try:
        payload = AGENT_RUNNERS[agent_name](query)
    except Exception as exc:
        error = str(exc)
    total_ms = round((perf_counter() - t0) * 1000, 2)

    return {
        "agent": agent_name,
        "query": query,
        "latency_ms": total_ms,
        "error": error,
        "payload": payload,
    }

def compare_agents(query: str, agents: list[str] | None = None) -> pd.DataFrame:
    selected = agents or list(AGENT_RUNNERS.keys())
    rows = []
    for name in selected:
        out = run_single_agent_test(name, query)
        payload = out.get("payload") or {}
        rows.append({
            "agent": name,
            "latency_ms": out["latency_ms"],
            "confidence_score": payload.get("confidence_score"),
            "confidence_reason": payload.get("confidence_reason"),
            "answer_preview": str(payload.get("answer", ""))[:140],
            "error": out.get("error"),
        })
    return pd.DataFrame(rows).sort_values(by="latency_ms", ascending=True)

In [ ]:
# Isolated agent testing — match_facts_agent removed, bigquery_agent is the data source.
query = "Portugal vs Morocco World Cup 2026"
agent_df = compare_agents(query, agents=["news", "sentiment", "prediction", "bigquery", "chat"])
display(agent_df)

# Deep dive bigquery agent directly
single = run_single_agent_test("bigquery", query)
print("Agent:", single["agent"])
print("Latency (ms):", single["latency_ms"])
print("Error:", single["error"])
print("Confidence:", (single["payload"] or {}).get("confidence_score"))
print("Reason:", (single["payload"] or {}).get("confidence_reason"))
print("\nAnswer:\n")
print((single["payload"] or {}).get("answer", ""))

## How To Use This For Workflow Improvement

1. Run the full workflow test with representative user messages.
2. Inspect step timing and output keys to find bottlenecks or weak transitions.
3. Compare isolated agent responses for the same query.
4. Focus improvements on:
   - slowest nodes
   - low confidence reasons
   - frequent errors in specific agents
   - mismatch between planner-selected agents and best-performing isolated agents